# Analyze a Document with Content Understanding

This notebook submits a publicly accessible document URL to the **invoice-analyzer-demo** analyzer created in `01_create_analyzer.ipynb` and prints the extracted structured fields.

> **Run `01_create_analyzer.ipynb` first** to ensure the analyzer exists.

In [1]:
%pip install azure-identity python-dotenv requests --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\w\repos\foundry-tools\.venv\Scripts\python.exe -m pip install --upgrade pip


In [2]:
import os
import json
import time
import requests
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  # loads .env from repo root

endpoint = os.environ["AZURE_CONTENT_UNDERSTANDING_ENDPOINT"].rstrip("/")
credential = DefaultAzureCredential()
token = credential.get_token("https://cognitiveservices.azure.com/.default").token
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json",
}

API_VERSION = "2025-11-01"
ANALYZER_ID = "invoiceAnalyzerDemo"

print("Client ready.")

Client ready.


In [3]:
# Use a publicly accessible sample invoice PDF
# Replace with your own document URL if needed
DOCUMENT_URL = "https://github.com/Azure-Samples/azure-ai-content-understanding-python/raw/refs/heads/main/data/invoice.pdf"

analyze_url = f"{endpoint}/contentunderstanding/analyzers/{ANALYZER_ID}:analyze?api-version={API_VERSION}"
body = {"inputs": [{"url": DOCUMENT_URL}]}

# Submit the analysis job
response = requests.post(analyze_url, headers=headers, json=body)
if not response.ok:
    print(f"Error {response.status_code}: {response.text}")
    response.raise_for_status()

# The service returns 202 Accepted with an Operation-Location header
operation_url = response.headers.get("Operation-Location")
if not operation_url:
    # Some API versions return the result immediately
    result = response.json()
    print("Result received immediately:")
    print(json.dumps(result, indent=2))
else:
    print(f"Analysis job submitted. Polling: {operation_url}")

Analysis job submitted. Polling: https://msftfoundrysdellalibera.services.ai.azure.com/contentunderstanding/analyzerResults/9ca829fa-ae6a-4af3-893f-de9f59c44db3?api-version=2025-11-01


In [4]:
# Poll until the analysis job completes
if operation_url:
    poll_headers = {"Authorization": f"Bearer {token}"}
    for attempt in range(30):
        poll_response = requests.get(operation_url, headers=poll_headers)
        poll_response.raise_for_status()
        status_data = poll_response.json()
        status = status_data.get("status", "unknown")
        print(f"  [{attempt + 1}] Status: {status}")
        if status.lower() in ("succeeded", "failed", "canceled"):
            break
        time.sleep(3)

    if status.lower() == "succeeded":
        result = status_data
        print("\nAnalysis succeeded!")
    else:
        raise RuntimeError(f"Analysis did not succeed. Final status: {status}")

  [1] Status: Running


  [2] Status: Succeeded

Analysis succeeded!


In [5]:
# Display extracted fields
print("\n=== Extracted Fields ===")
try:
    fields = result["result"]["contents"][0]["fields"]
    for field_name, field_value in fields.items():
        value = (
            field_value.get("valueString")
            or field_value.get("valueNumber")
            or field_value.get("valueDate")
            or "(not found)"
        )
        print(f"  {field_name}: {value}")
except (KeyError, IndexError):
    print("Could not parse fields from result. Raw result:")
    print(json.dumps(result, indent=2))


=== Extracted Fields ===
  InvoiceId: INV-100
  VendorName: CONTOSO LTD.
  InvoiceDate: 2019-11-15
  TotalAmount: 610
